https://msea.readthedocs.io/en/latest/tutorial.html

In [5]:
import msea
# %pip install msea
from msea import SetLibrary
import numpy
import scipy
import pandas as pd
from pprint import pprint
# %pip install biom-format
from biom import load_table  # reqired for parsing BIOM formated dataset
from skbio.stats.composition import ancom
# %pip install scikit-bio
import re


In [32]:
microbiome_data = pd.read_excel("../csv_files/Xero-Microbiome RA data patient matched AH 07 23 25 females ROC above 0.7.xlsx")

In [19]:
microbiome_data

,Group,Condition,Site,Taxa,Other;Other,g__Porphyromonas;s__sp13375,g__Porphyromonas;s__sp13376,g__Streptococcus;s__australis,g__Streptococcus;s__australis-sanguinis,g__Streptococcus;s__mutans,g__Veillonella;s__atypica,k__Bacteria;p__NA;c__NA;o__NA;f__NA;g__NA;s__sp19816,g__Haemophilus;s__parainfluenzae,g__Haemophilus;s__pittmaniae
0,Xerostomic,Autoimmune,S,XOB009.S,0.014811,0.000123,0.017486,0.006436,0.0,0.000000,0.018959,0.000000,0.037097,0.000000
1,Xerostomic,Autoimmune,B,XOB009.B,0.010084,0.000811,0.020203,0.012126,0.0,0.000103,0.009630,0.000000,0.142916,0.000000
2,Xerostomic,Autoimmune,T,XOB009.T,0.021316,0.000000,0.027809,0.007084,0.0,0.000000,0.011404,0.000000,0.068349,0.000000
3,Xerostomic,Autoimmune,T,XOB012.T,0.012974,0.005788,0.016771,0.000642,0.0,0.000000,0.026237,0.000000,0.011216,0.000000
4,Xerostomic,Autoimmune,B,XOB012.B,0.000941,0.001040,0.003309,0.000166,0.0,0.000315,0.000575,0.000000,0.001466,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,Xerostomic,Medication,S,XOB064.S,0.002698,0.000000,0.011398,0.018180,0.0,0.000000,0.010011,0.000000,0.026936,0.000000
113,Xerostomic,Medication,B,XOB064.B,0.000492,0.000081,0.005652,0.000705,0.0,0.000000,0.000910,0.000000,0.010585,0.000000
114,Xerostomic,Medication,S,XOB067.S,0.004753,0.000807,0.002199,0.000192,0.0,0.006375,0.078955,0.000215,0.013489,0.001246
115,Xerostomic,Medication,B,XOB067.B,0.007442,0.003342,0.005133,0.000000,0.0,0.002221,0.032537,0.000215,0.022090,0.001765


- read the file
- split into counts and metadata

In [33]:
cols = list(microbiome_data.columns)
microbiome_data.reindex(columns=cols[-3:]+cols[:-3]) ## Group, Condition, Site

microbiome_data = microbiome_data.iloc[:, :-1]


cols = list(microbiome_data.columns)
metadata = microbiome_data[ cols[0:4] ]

data_cols = microbiome_data[ cols[5:-1] ]

## aggregate to the genus level
data_cols.columns = [col.split(";")[-2] if "g__" in col else col for col in data_cols.columns]
data_cols = data_cols.T.groupby(data_cols.columns).sum().T

data_cols = data_cols.loc[:,["g__Porphyromonas", "g__Haemophilus", "g__Streptococcus"]]
# data_cols = data_cols.loc[:,["g__Porphyromonas", "g__Haemophilus", "g__Streptococcus", "g__Rothia"]]


data_cols



,g__Porphyromonas,g__Haemophilus,g__Streptococcus
0,0.017608,0.037097,0.006436
1,0.021014,0.142916,0.012229
2,0.027809,0.068349,0.007084
3,0.022560,0.011216,0.000642
4,0.004349,0.001466,0.000481
...,...,...,...
112,0.011398,0.026936,0.018180
113,0.005733,0.010585,0.000705
114,0.003007,0.013489,0.006567
115,0.008475,0.022090,0.002221


Next, we perform DA analysis using ANCOM to identify DA microbes:

In [35]:

ancom_df, percentile_df = ancom(data_cols + 1,  # adding pseudocounts
                            metadata['Group'],
                            alpha=0.1,
                            p_adjust='holm-bonferroni')

microbes_DA = ancom_df.loc[ancom_df['Signif']].index

ancom_df.loc[ancom_df['Signif']].to_csv("../output/without_men_10percent_filtered/msea_significantRF/significant_bacteria.csv") ## save the ancom_df

print(microbes_DA)

# retain genus names
microbes_DA = filter(None, [s.split('; ')[-1].lstrip('g__') for s in microbes_DA])

# convert to set
microbes_DA = set(microbes_DA)


percentile_df
ancom_df
# microbes_DA
    

Index(['g__Porphyromonas', 'g__Haemophilus', 'g__Streptococcus'], dtype='object')


,W,Signif
g__Porphyromonas,2,True
g__Haemophilus,2,True
g__Streptococcus,2,True


Finally, we can perform MSEA for DA microbes we just identified:

In [15]:
gmt_filepath = \
    'https://bitbucket.org/wangz10/msea/raw/aee6dd184e9bde152b4d7c2f3c7245efc1b80d23/msea/data/human_genes_associated_microbes/set_library.gmt'

d_gmt = msea.read_gmt(gmt_filepath)

# Look at a couple of reference sets in the library
pprint(list(d_gmt.items())[:3])

[('A2M', {'Sodalis', 'Salmonella', 'Borrelia', 'Pseudomonas', 'Azomonas'}),
 ('AAAS',
  {'Colwellia',
   'Deinococcus',
   'Idiomarina',
   'Neisseria',
   'Pseudidiomarina',
   'Pseudoalteromonas'}),
 ('AACS',
  {'Acetobacter',
   'Acinetobacter',
   'Azomonas',
   'Corynebacterium',
   'Enterobacter',
   'Klebsiella',
   'Mycobacterium',
   'Mycoplasma',
   'Pseudomonas',
   'Sodalis',
   'Staphylococcus',
   'Streptomyces',
   'Tetragenococcus'})]


In [13]:
# this can be done using the `msea.enrich` function
msea_result = msea.enrich(microbes_DA, d_gmt=d_gmt, universe=1000)
# check the top enriched reference microbe-sets
print(msea_result.head())

           oddsratio    pvalue    qvalue  \
term                                       
RAB14      62.500000  0.000087  0.112400   
BCL6       62.500000  0.001287  0.138385   
CXCL10     15.306122  0.003430  0.138385   
PDCD1LG2  100.000000  0.000609  0.138385   
CXCL8      16.666667  0.002746  0.138385   

                                             shared  n_shared  
term                                                           
RAB14     [Porphyromonas, Veillonella, Haemophilus]         3  
BCL6                   [Porphyromonas, Haemophilus]         2  
CXCL10    [Porphyromonas, Veillonella, Haemophilus]         3  
PDCD1LG2               [Porphyromonas, Haemophilus]         2  
CXCL8     [Porphyromonas, Veillonella, Haemophilus]         3  


Step 2: perform MSEA with adjustment of expected ranks for reference sets. Sometimes certain reference microbe-sets in a library are more likely to be enriched by chance. We can adjust this by empirically estimating the null distributions of the ranks of the reference sets, then uses z-score to quantify if observed ranks are significantly different from the expected ranks.

To do that, it is easier through the [`SetLibrary`](https://msea.readthedocs.io/en/latest/api_docs.html#msea.SetLibrary) class:

The [`SetLibrary.get_empirical_ranks()`](https://msea.readthedocs.io/en/latest/api_docs.html#msea.SetLibrary.get_empirical_ranks) method helps compute the expected ranks and store the means and standard deviations of the ranks from the null distribution:


In [16]:
set_library = SetLibrary.load(gmt_filepath)
set_library.get_empirical_ranks(n=20)
print(set_library.rank_means.shape, set_library.rank_stds.shape)



Calculating empirical ranks for each set...
Number of unique microbes: 566


100%|███████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:06<00:00,  3.27it/s]

(1286,) (1286,)


After that, we can perform MSEA with this adjustment:


In [17]:

msea_result_adj = set_library.enrich(
    microbes_DA, adjust=True, universe=1000)
print(msea_result_adj.head())

msea_result_adj

           oddsratio    pvalue    qvalue    zscore  combined_score  \
term                                                                 
PDF        44.444444  0.002541  0.019682 -4.812679       28.757432   
TNFRSF11B  66.666667  0.000089  0.013158 -2.765358       25.784493   
PAK4       95.238095  0.000694  0.013677 -3.329739       24.217271   
LAMP1      40.000000  0.000340  0.013677 -2.649645       21.158122   
LCN2       33.333333  0.000553  0.013677 -2.722185       20.417816   

                                                shared  n_shared  
term                                                              
PDF                       [Streptococcus, Haemophilus]         2  
TNFRSF11B  [Streptococcus, Porphyromonas, Haemophilus]         3  
PAK4                      [Streptococcus, Haemophilus]         2  
LAMP1      [Streptococcus, Porphyromonas, Haemophilus]         3  
LCN2       [Streptococcus, Porphyromonas, Haemophilus]         3  


,oddsratio,pvalue,qvalue,zscore,combined_score,shared,n_shared
term,,,,,,,
PDF,44.444444,0.002541,0.019682,-4.812679,28.757432,"[Streptococcus, Haemophilus]",2
TNFRSF11B,66.666667,0.000089,0.013158,-2.765358,25.784493,"[Streptococcus, Porphyromonas, Haemophilus]",3
PAK4,95.238095,0.000694,0.013677,-3.329739,24.217271,"[Streptococcus, Haemophilus]",2
LAMP1,40.000000,0.000340,0.013677,-2.649645,21.158122,"[Streptococcus, Porphyromonas, Haemophilus]",3
LCN2,33.333333,0.000553,0.013677,-2.722185,20.417816,"[Streptococcus, Porphyromonas, Haemophilus]",3
...,...,...,...,...,...,...,...
IL15,29.411765,0.000772,0.013719,2.444923,-17.523124,"[Streptococcus, Porphyromonas, Haemophilus]",3
MAP3K8,111.111111,0.000542,0.013677,2.373388,-17.848788,"[Porphyromonas, Haemophilus]",2
LY96,50.000000,0.000189,0.013677,2.221648,-19.050310,"[Streptococcus, Porphyromonas, Haemophilus]",3


In [19]:
msea_result_adj.to_csv("../output/without_men/msea_significantRF/msea_results.csv")